## Path configuration

In [ ]:
from pathlib import Path
import os

PROJECT_NAME = "MALDIAlign"

cwd = Path().resolve()

# Walk upwards until we find the project folder
target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

# If the project folder is found and we are not already there, then change cwd
if target is not None and target != cwd:
    os.chdir(target)

print("Working directory:", os.getcwd())

## Imports

In [ ]:
import os
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.load_config import load_config
from utils.load_data import load_pkl
from utils.eval import eval_model
from utils.viz import *

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

## Data preparation

### Data loading

In [ ]:
cfg = load_config()
driams_pkl = cfg["data"]["DRIAMS_REDUCED_PKL"]

In [ ]:
driams = load_pkl(driams_pkl)

In [ ]:
data, label, meta = driams["data"], driams["label"], driams["meta"]

In [ ]:
meta = pd.DataFrame.from_records(list(meta))

### Filter data and construct final dataset

In [ ]:
maskA = np.where(meta["hospital"].values == "DRIAMS_A")[0]
dataA, labelA, metaA = data[maskA], label[maskA], meta.iloc[maskA]

maskD = np.where(meta["hospital"].values == "DRIAMS_D")[0]
dataD, labelD, metaD = data[maskD], label[maskD], meta.iloc[maskD]

In [ ]:
# Downsample DRIAMS-A
from sklearn.model_selection import train_test_split

_, dataA_sub, _, labelA_sub, _, metaA_sub = train_test_split(
    dataA,
    labelA,
    metaA,
    test_size=len(dataD),
    random_state=42,
    stratify=labelA)

In [ ]:
# Concatenate data and labels
data_final = np.vstack([dataA_sub, dataD])
label_final = np.concatenate([labelA_sub, labelD])
meta_final  = pd.concat([metaA_sub, metaD], ignore_index=True)

In [ ]:
species, counts = np.unique(labelA_sub, return_counts=True)
for sp, n in zip(species, counts):
    print(f"{sp}: {n}")

In [ ]:
species, counts = np.unique(labelD, return_counts=True)
for sp, n in zip(species, counts):
    print(f"{sp}: {n}")

### Construct dataloaders

In [ ]:
domain_map = {"DRIAMS_A": 0, "DRIAMS_D": 1}
domain_ids = meta_final["hospital"].map(domain_map).values

In [ ]:
X_train, X_val, y_train, y_val, domain_train, domain_val = train_test_split(
    data_final, label_final, domain_ids,
    test_size=0.2, random_state=42, stratify=label_final
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val, dtype=torch.float32)

In [ ]:
domain_train_tensor = torch.tensor(domain_train, dtype=torch.long)
domain_val_tensor   = torch.tensor(domain_val, dtype=torch.long)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, domain_train_tensor)
val_dataset   = TensorDataset(X_val_tensor, domain_val_tensor)

In [ ]:
targets = domain_train_tensor
class_counts = np.bincount(targets)
class_weights = 1. / class_counts
sample_weights = class_weights[targets]

sample_weights = torch.from_numpy(sample_weights).double()

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    sampler=sampler,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

## Architecture

In [ ]:
class InvariantEncoder(nn.Module):
  def __init__(self, input_dim, latent_dim):
    super().__init__()

    self.net = nn.Sequential(
      nn.Linear(input_dim, 512),
          nn.ReLU(),
          nn.Linear(512, 256),
          nn.ReLU()
    )

    self.mu = nn.Linear(256, latent_dim)
    self.logvar = nn.Linear(256, latent_dim)

  def forward(self, x):
    hidden_rep = self.net(x)
    mu = self.mu(hidden_rep)
    logvar = self.logvar(hidden_rep)
    logvar = torch.clamp(logvar, -6, 6)
    return mu, logvar

In [ ]:
class ConditionalDecoder(nn.Module):
  def __init__(self, latent_dim, output_dim, cond_dim):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(latent_dim + cond_dim, 256),
        nn.ReLU(),
        nn.Linear(256, 512),
        nn.ReLU(),
        nn.Linear(512, output_dim)
    )

  def forward(self, z, c):
    h = torch.cat([z, c], dim=1)
    x_recon = self.net(h)
    return x_recon

In [ ]:
class InvariantCVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, cond_dim):
        super().__init__()
        self.encoder = InvariantEncoder(input_dim, latent_dim)
        self.decoder = ConditionalDecoder(latent_dim, input_dim, cond_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, c):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z, c)
        return x_recon, mu, logvar

In [ ]:
class InvariantCVAE_Extended(InvariantCVAE):
    def __init__(self, input_dim, latent_dim, cond_dim, epochs=100, lr=1e-4, annealing_epochs=50):
        super().__init__(input_dim, latent_dim, cond_dim)
        self.epochs = epochs
        self.lr = lr
        self.annealing_epochs = annealing_epochs
        self.optimizer = optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-5)
        self.criterion = nn.MSELoss()
        self.loss_during_training = []
        self.reconstruc_during_training = []
        self.KL_during_training = []

    def loss_function(self, x, x_recon, mu, logvar, beta):
        recon_loss = self.criterion(x_recon, x)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        total_loss = recon_loss + beta * kl_loss
        return total_loss, recon_loss, kl_loss

    def trainloop(self, trainloader, validloader, device):
        self.to(device)
        for epoch in range(self.epochs):
            beta = min(1.0, (epoch + 1) / self.annealing_epochs)
            self.train()
            train_total, train_recon, train_kl = 0, 0, 0

            for x, domain_id in trainloader:
                x = x.to(device)
                # One-hot encode domain
                c = nn.functional.one_hot(domain_id, num_classes=2).float().to(device)
                self.optimizer.zero_grad()
                x_recon, mu, logvar = self.forward(x, c)
                loss, recon, kl = self.loss_function(x, x_recon, mu, logvar, beta)
                loss.backward()
                self.optimizer.step()
                train_total += loss.item()
                train_recon += recon.item()
                train_kl += kl.item()

            train_total /= len(trainloader)
            train_recon /= len(trainloader)
            train_kl /= len(trainloader)

            # Validation
            self.eval()
            val_loss, val_recon, val_kl = 0, 0, 0
            with torch.no_grad():
                for x, domain_id in validloader:
                    x = x.to(device)
                    c = nn.functional.one_hot(domain_id, num_classes=2).float().to(device)
                    x_recon, mu, logvar = self.forward(x, c)
                    loss, recon, kl = self.loss_function(x, x_recon, mu, logvar, beta)
                    val_loss += loss.item()
                    val_recon += recon.item()
                    val_kl += kl.item()
            val_loss /= len(validloader)
            val_recon /= len(validloader)
            val_kl /= len(validloader)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{self.epochs} | "
                      f"[Train] Loss: {train_total:.4f} | Recon: {train_recon:.4f} | KL: {train_kl:.4f} || "
                      f"[Val] Loss: {val_loss:.4f} | Recon: {val_recon:.4f} | KL: {val_kl:.4f}")

            self.loss_during_training.append((train_total, val_loss))
            self.reconstruc_during_training.append((train_recon, val_recon))
            self.KL_during_training.append((train_kl, val_kl))

## Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

my_invariant_cvae = InvariantCVAE_Extended(
    input_dim=X_train.shape[1],
    latent_dim=64,
    cond_dim=2,
    epochs=200,
    annealing_epochs=100
).to(device)

In [ ]:
my_invariant_cvae.trainloop(train_loader, val_loader, device)

In [ ]:
torch.save(my_invariant_cvae.state_dict(), "cvae_invariant.pth")

## Plot results

In [ ]:
plot_model_metrics(my_invariant_cvae, "Invariant cVAE")

## t-SNE

In [ ]:
X_all = scaler.transform(data_final)
X_all_tensor = torch.tensor(X_all, dtype=torch.float32)
domain_train_tensor = torch.tensor(domain_ids, dtype=torch.long)

In [ ]:
all_dataset = TensorDataset(X_all_tensor, domain_train_tensor)
all_loader  = DataLoader(all_dataset, batch_size=256, shuffle=False)

Get latent means for all samples

In [ ]:
mus_all = eval_model(my_invariant_cvae, all_loader, device, use_domain=False)

Global t-SNE

In [ ]:
tsne_df = compute_tsne_df(mus_all, label_final, meta_final)
plot_tsne_global(tsne_df, False)

In [ ]:
plot_tsne_global(tsne_df, True)

t-SNE for each species separately

In [ ]:
df_all, tsne_results = compute_tsne_per_species(mus_all, label_final, meta_final)
plot_tsne_species(df_all, tsne_results)